# 02 - Pipeline validation
Row counts, expectations and sample gold output from the Lakeflow declarative pipeline.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
from export_evidence import pipeline_id
pid, name = pipeline_id(); print(pid, name)
ups = ws().pipelines.list_updates(pid, max_results=5).updates
pd.DataFrame([{'update_id': u.update_id, 'state': u.state.value, 'full_refresh': u.full_refresh} for u in ups])

41d741ff-7755-44b7-9cfc-bb305a327764 [dev erica_yuen] matter-intelligence-pipeline


,update_id,state,full_refresh
0,024cf0f4-cce5-4efe-9e7e-239d1c79fab9,COMPLETED,False
1,1ed2c816-5a65-424a-ae16-fa85ebae177f,COMPLETED,False
2,ba94fa9a-3741-4a89-b347-726ae348298b,COMPLETED,True
3,880eb606-e30b-4035-9677-4aa559a7fcac,COMPLETED,False
4,bd2a970d-5353-4f19-b869-45baa764f83a,COMPLETED,False


In [3]:
from export_evidence import TABLES
sql(' UNION ALL '.join("SELECT '%s' t, count(*) n FROM ${catalog}.%s" % (t, t) for t in TABLES))

,t,n
0,curated.gold_documents,100
1,curated.gold_clauses,775
2,curated.gold_risk_flags,293
3,raw.bronze_documents,100
4,curated.silver_parsed,100
5,curated.int_doc_classified,100
6,curated.int_doc_extracted,100
7,curated.int_clauses,775
8,curated.int_risk_flags,293
9,curated.gold_cost_metrics,5


In [4]:
sql(f"""WITH ev AS (SELECT origin.update_id, explode(from_json(details:flow_progress.data_quality.expectations::string,
 'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) e
 FROM event_log('{pid}') WHERE event_type='flow_progress' AND details:flow_progress.data_quality.expectations IS NOT NULL)
SELECT update_id, e.dataset, e.name, sum(e.passed_records) passed, sum(e.failed_records) failed FROM ev GROUP BY ALL ORDER BY 1,2,3""")

,update_id,dataset,name,passed,failed
0,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.int_clauses,confidence_not_null,166,0
1,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.silver_parsed,parse_ok,20,0
2,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.silver_parsed,text_min_length,20,0
3,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.raw.bronze_documents,has_content,20,0
4,ba94fa9a-3741-4a89-b347-726ae348298b,serverless_stable_oj7eie_catalog.curated.int_clauses,confidence_not_null,775,0
5,ba94fa9a-3741-4a89-b347-726ae348298b,serverless_stable_oj7eie_catalog.curated.silver_parsed,parse_ok,100,0
6,ba94fa9a-3741-4a89-b347-726ae348298b,serverless_stable_oj7eie_catalog.curated.silver_parsed,text_min_length,100,0
7,ba94fa9a-3741-4a89-b347-726ae348298b,serverless_stable_oj7eie_catalog.raw.bronze_documents,has_content,100,0
8,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.curated.int_clauses,confidence_not_null,167,0
9,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.curated.silver_parsed,parse_ok,20,0


In [5]:
sql("SELECT doc_id, page_count, parse_status, length(full_text) chars, parse_error FROM ${catalog}.curated.silver_parsed ORDER BY doc_id")

,doc_id,page_count,parse_status,chars,parse_error
0,000982989fb935a8457eea97ff59e3b5562c41faa8ebd4e0936ce6e2543659f8,3,ok,6636,None
1,007493cb2c881bc464f4ee102044914082f89a8018fced6482358426d9bb7340,10,ok,34812,None
2,05e6e15e6f04f9368ad8b749f477902d3f767c42a92daba470c4df9ea1e82ba1,27,ok,86765,None
3,158974d5f50f2300a0ecbce4079db8939d2313310d97281dae51f3e66da880be,1,ok,2238,None
4,172aa8dd1f0d9e568c1435f033d36f8e5612625c3bbce69ab7ad652f6282daea,56,ok,111268,None
...,...,...,...,...,...
95,f364a29a94fce12d385b611375a45e159d12fe1f3bbc1ba939f936b8cc0fb2fe,14,ok,29432,None
96,f5eb4ec919b16dfbcb00ba3c87fa23448b1bd1f84308db28f92c8416dbe63ddc,13,ok,19844,None
97,f62bad5586ec28f1260a5b832cb62bc34d60b1536367096669a564d13ffac3e2,1,ok,1684,None
98,fb885b727c026bf65e64eb1a21819b0ec9864094459d2cd4afcce3da294287ef,52,ok,119641,None


In [6]:
sql("SELECT contract_type, count(*) docs, round(avg(contract_type_confidence),3) conf FROM ${catalog}.curated.gold_documents GROUP BY 1 ORDER BY 2 DESC")

,contract_type,docs,conf
0,MSA,23,0.767
1,License,22,0.791
2,Distribution,22,0.776
3,Other,21,0.775
4,Supply,9,0.788
5,Employment,2,0.78
6,NDA,1,0.75


In [7]:
sql("SELECT clause_type, count(*) n, round(avg(confidence),3) conf FROM ${catalog}.curated.gold_clauses GROUP BY 1 ORDER BY 1")

,clause_type,n,conf
0,assignment_restriction,74,0.968
1,auto_renewal,61,0.95
2,effective_date,89,0.969
3,exclusivity,47,0.977
4,expiration_date,44,0.96
5,governing_law,78,0.987
6,indemnification,65,0.961
7,liability_cap,36,0.979
8,non_compete,36,0.988
9,notice_period_days,25,0.965


In [8]:
sql("SELECT clause_type, normalized_value, round(confidence,2) conf, left(source_snippet, 150) snippet FROM ${catalog}.curated.gold_clauses WHERE clause_type IN ('governing_law','auto_renewal','uncapped_liability','notice_period_days') ORDER BY doc_id, clause_type LIMIT 12")

,clause_type,normalized_value,conf,snippet
0,auto_renewal,true,1.0,with this Agreement. This Agreement shall automatically extend for an additional two (2) years on the same terms her...
1,governing_law,California,1.0,g. Governing Law and Forum. This Agreement will for all purposes be governed by and interpreted in accordance with t...
2,uncapped_liability,true,1.0,"c. LIMITATIONS ON LIABILITY/NO INJUNCTIVE RELIEF. EXCEPT IN CASES OF GROSS NEGLIGENCE, WILLFUL\nMISCONDUCT OR FRAUD,..."
3,auto_renewal,false,1.0,This term of this Agreement and the rights and obligations of the Parties hereto shall commence as of the Effective ...
4,governing_law,Province Of Ontario,1.0,"This Agreement shall be governed by, and construed and enforced in accordance with, the laws in force in the Provinc..."
5,uncapped_liability,true,1.0,"Notwithstanding Sections 17(a) and 17(b), neither Party excludes or limits any liability for:\n(i) personal injury o..."
6,governing_law,New York,1.0,"Section 9.6 <u>Governing Law, Jurisdiction, Venue.</u> THIS AGREEMENT SHALL BE GOVERNED BY AND INTERPRETED IN ACCORD..."
7,governing_law,"R.o.c. (republic Of China, Taiwan)",1.0,herein agree to take the Miaoli District Court of Taiwan as the competent court of first instance pursuant to the la...
8,governing_law,Delaware,1.0,6.1 Governing Law. This Agreement shall be governed by and interpreted in accordance with the laws of the State of D...
9,auto_renewal,false,1.0,(a) This Agreement is effective as of the date hereof and shall remain in full force and effect for the period (the ...


In [9]:
sql("SELECT severity, rule_id, count(*) flags, count(explanation) explained FROM ${catalog}.curated.gold_risk_flags GROUP BY ALL ORDER BY 1,2")

,severity,rule_id,flags,explained
0,high,R01,26,26
1,high,R02,36,36
2,low,R06,46,0
3,low,R07,37,0
4,low,R08,74,0
5,medium,R03,17,17
6,medium,R04,10,10
7,medium,R05,47,47


In [10]:
sql("SELECT rule_id, severity, left(explanation, 300) explanation FROM ${catalog}.curated.gold_risk_flags WHERE explanation IS NOT NULL LIMIT 5")

,rule_id,severity,explanation
0,R02,high,"This clause triggers the playbook rule because the Member ""covenants and agrees that during the term of the Franchis..."
1,R03,medium,"This clause triggers the auto_renewal playbook rule because the agreement ""automatically renewing annually thereafte..."
2,R05,medium,"This clause triggers the playbook rule because it explicitly requires that ""Talent will not endorse or make any appe..."
3,R02,high,"This clause triggers the rule because it explicitly restricts the Talent party from competing by providing that ""Tal..."
4,R05,medium,"This clause triggers the playbook rule because Party A and Party B ""agree to have long-term cooperation for current ..."


In [11]:
sql("SELECT * FROM ${catalog}.curated.gold_cost_metrics ORDER BY run_id, stage")

,run_id,stage,docs,pages,input_tokens,output_tokens,est_cost_usd,cost_per_1k_docs
0,20261118,classify,100,NaN,429146,3000,0.2191,2.19
1,20261118,explain,63,NaN,65915,13282,0.1323,1.32
2,20261118,extract,100,NaN,966279,66425,0.5828,5.83
3,20261118,parse,100,1986,NaN,NaN,9.93,99.3
4,20261118,total,100,1986,1461340,82707,10.8642,108.64
